In [ ]:
!pip install ansys-mapdl-core

In [ ]:
from ansys.mapdl.core import launch_mapdl

In [ ]:
mapdl = launch_mapdl()
print(mapdl)

In [ ]:
mapdl = launch_mapdl()
mapdl.clear()
mapdl.prep7()

In [ ]:
mapdl.exit()

In [ ]:
#~~~~~~~~For Job Scheduling~~~~~~~~~~~~~#
import os
path=os.getcwd()
jname='user_jobname'
mapdl=launch_mapdl(run_location=path,override=True)
mapdl.ignore_errors=True

In [ ]:
#~~~~~~~~For Converting APDL scripts to MAPDL~~~~~~~~~~~~~#
import ansys.mapdl.core as pymapdl
input_file = '<path_to_your_workspace>\/MultiChiplet_files/geometry_25D.scscript'
pyscript = '<path_to_your_workspace>/Ansys CAD/python_files/geometry_25D.py'
pymapdl.convert_script(input_file,pyscript)

In [ ]:
##############################################################################
# 2) Define your geometry data (same as your SpaceClaim script)
##############################################################################
layers = [
    {"name": "Substrate",    "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "C4_Bumps",     "length": 30.0, "width": 30.0, "thickness": 0.1},
    {"name": "Interposer",   "length": 25.0, "width": 25.0, "thickness": 0.3},
    {"name": "MicroBumps",   "length": 25.0, "width": 25.0, "thickness": 0.05},
    {"name": "TIM_pads",     "length": 30.0, "width": 30.0, "thickness": 0.05},
    {"name": "HeatSpreader", "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "TIM2_pads",    "length": 35.0, "width": 35.0, "thickness": 0.05},
    # Instead of a single 2 mm block, we will build a heatsink with fins.
    {"name": "HeatSink",     "length": 40.0, "width": 40.0, "thickness": 2.0},
]

chiplets = [
    {"name": "Chiplet_1", "x_offset": 5.0,  "y_offset": 5.0,  "length": 5.0, "width": 5.0, "thickness": 0.2},
    {"name": "Chiplet_2", "x_offset": 10.0, "y_offset": 5.0,  "length": 4.0, "width": 4.0, "thickness": 0.2},
    {"name": "Chiplet_3", "x_offset": 15.0, "y_offset": 5.0,  "length": 6.0, "width": 6.0, "thickness": 0.2},
]

##############################################################################
# 3) Figure out how to center them (like your SpaceClaim script)
##############################################################################
max_length = max(layer["length"] for layer in layers)
max_width  = max(layer["width"]  for layer in layers)

In [ ]:
import math
import sys

current_z = 0.0

for i, layer in enumerate(layers):
    layer_name = layer["name"]
    L_mm = layer["length"]
    W_mm = layer["width"]
    T_mm = layer["thickness"]
    
    x1 = (max_length - L_mm) / 2.0
    y1 = (max_width - W_mm) / 2.0
    x2 = x1 + L_mm
    y2 = y1 + W_mm
    z1 = current_z
    z2 = current_z + T_mm
    
    print(f"Creating layer {i+1}: {layer_name}")
    print(f"Coordinates: x1={x1}, x2={x2}, y1={y1}, y2={y2}, z1={z1}, z2={z2}")
    
    if layer_name == "HeatSink":
        base_thickness_mm = 1.0
        fin_height_mm = 6.0
        mapdl.block(x1, x2, y1, y2, z1, z1 + base_thickness_mm)
        mapdl.run("! Created HeatSink_Base volume")
        
        fin_count = 13
        fin_thickness = 1.0
        fin_gap = 2.0
        total_x = x2 - x1
        margin_x = 2.0
        region_for_fins_x = total_x - 2 * margin_x
        fin_start_x = x1 + margin_x
        current_x = fin_start_x
        
        for fin_index in range(fin_count):
            if current_x + fin_thickness > x2 - margin_x:
                break
            fin_x1 = current_x
            fin_x2 = current_x + fin_thickness
            fin_y1 = y1
            fin_y2 = y2
            fin_z1 = z1 + base_thickness_mm
            fin_z2 = z1 + base_thickness_mm + fin_height_mm
            mapdl.block(fin_x1, fin_x2, fin_y1, fin_y2, fin_z1, fin_z2)
            mapdl.run("! Created HeatSink_Fin #{}".format(fin_index + 1))
            current_x += (fin_thickness + fin_gap)
        
        current_z += T_mm
    
    elif layer_name == "MicroBumps":
        # Create the MicroBumps layer
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created MicroBumps layer")
        
        # Advance the stacking height by the thickness of the MicroBumps layer
        current_z += T_mm
        
        # Create the chiplets on top of the MicroBumps layer
        max_chip_thickness_mm = 0.0
        for chip in chiplets:
            chip_name = chip["name"]
            c_offset_x = chip["x_offset"]
            c_offset_y = chip["y_offset"]
            c_length_mm = chip["length"]
            c_width_mm = chip["width"]
            c_thk_mm = chip["thickness"]
            chip_x1 = x1 + c_offset_x
            chip_y1 = y1 + c_offset_y
            chip_x2 = chip_x1 + c_length_mm
            chip_y2 = chip_y1 + c_width_mm
            chip_z1 = current_z
            chip_z2 = chip_z1 + c_thk_mm
            mapdl.block(chip_x1, chip_x2, chip_y1, chip_y2, chip_z1, chip_z2)
            mapdl.run("! Created chip: '{}'".format(chip_name))
            if c_thk_mm > max_chip_thickness_mm:
                max_chip_thickness_mm = c_thk_mm
        
        # Advance the stacking height by the tallest chip
        current_z += max_chip_thickness_mm
    
    else:
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created layer volume for '{}'".format(layer_name))
        current_z += T_mm
    
    # Plot the geometry after each layer is created
    

mapdl.vplot()
print("Geometry creation done. You can now inspect the volumes in MAPDL.")
# mapdl.exit()  # close if desired

In [ ]:
##############################################################################
# 5) (Optional) Define element types, mesh, etc.
##############################################################################
# For example:
mapdl.et(1, "SOLID185")   # 3D structural brick
mapdl.vmesh("ALL")        # mesh all volumes

# Ready for loads, boundary conditions, solve, etc.
mapdl.finish()
mapdl.run("/SOLU")
# e.g. apply constraints, loads, then:
# mapdl.solve()
# mapdl.finish()

##############################################################################
# 6) End or keep interactive
##############################################################################
print("Geometry creation done. You can now inspect the volumes in MAPDL.")
# mapdl.exit()  # close if desired


In [ ]:
%pip install ipywidgets
!pip install -U ipywidgets==7.7.1

In [ ]:
mapdl.vsbv(1,'all')
#mapdl.geometry.areas.plot() 
mapdl.vplot()

In [ ]:
##############################################################################
# Example: Material Property Definitions
##############################################################################
# Suppose we define a dictionary of materials and their properties
# for a thermo-mechanical analysis. The keys below are just examples.
# 
# NOTE on "volumetric heat capacity" vs "mass-based specific heat":
#   - ANSYS MAPDL's MP,C expects J/(kg*K) (mass-based).
#   - If you have volumetric heat capacity in J/(m^3*K),
#     you can do:  volumetric_C / density = "mass-based specific heat".
#
# For coefficient of thermal expansion (CTE), ALPX is typically in
# units of 1/°C or 1/K. 
##############################################################################
mapdl.prep7() #ReEnter Preprocessor

material_data = {
    "Substrate": {
        "mat_id": 1,                  # Material ID used in MAPDL
        "E": 3.0e10,                  # Young's modulus (Pa) for example
        "nu": 0.35,                   # Poisson's ratio
        "dens": 2.5e3,               # Density (kg/m^3)
        "alpha": 1.0e-5,             # CTE (1/K)
        "kxx": 10.0,                  # Thermal conductivity (W/m-K)
        "cp": 800.0,                  # Specific heat capacity (J/kg-K)
    },
    "C4_Bumps": {
        "mat_id": 2,
        "E": 5.0e10,
        "nu": 0.33,
        "dens": 8.9e3,
        "alpha": 2.3e-5,
        "kxx": 40.0,
        "cp": 200.0,
    },
    "Interposer": {
        "mat_id": 3,
        "E": 2.5e10,
        "nu": 0.34,
        "dens": 2.7e3,
        "alpha": 1.2e-5,
        "kxx": 20.0,
        "cp": 700.0,
    },
    "MicroBumps": {
        "mat_id": 4,
        "E": 2.0e10,
        "nu": 0.32,
        "dens": 2.7e3,
        "alpha": 2.2e-5,
        "kxx": 25.0,
        "cp": 210.0,
    },
    "TIM_pads": {
        "mat_id": 5,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSpreader": {
        "mat_id": 6,
        "E": 7.0e10,
        "nu": 0.29,
        "dens": 8.9e3,
        "alpha": 1.7e-5,
        "kxx": 400.0,
        "cp": 385.0,
    },
    "TIM2_pads": {
        "mat_id": 7,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSink": {
        "mat_id": 8,
        "E": 6.9e10,
        "nu": 0.33,
        "dens": 2.7e3,
        "alpha": 2.4e-5,
        "kxx": 205.0,
        "cp": 900.0,
    },
    "Chiplet": {
        "mat_id": 9,
        "E": 1.0e11,
        "nu": 0.28,
        "dens": 2.33e3,
        "alpha": 2.6e-6,
        "kxx": 149.0,
        "cp": 700.0,
    },
}

# Loop through the dictionary and define each material in MAPDL
for mat_name, props in material_data.items():
    mid   = props["mat_id"]
    E     = props["E"]
    nu    = props["nu"]
    dens  = props["dens"]
    alpha = props["alpha"]
    k     = props["kxx"]
    cp    = props["cp"]
    
    # Set active material ID
    mapdl.mptemp("", 0)      # reset temperature table
    mapdl.mptemp(1, 25)      # define reference temp = 25 °C (example)
    mapdl.mp("EX",   mid, E)
    mapdl.mp("PRXY", mid, nu)
    mapdl.mp("DENS", mid, dens)
    
    # Thermal expansion
    # (For isotropic expansion, you can use ALPX. 
    #  If you want temperature-dependent data, you'd define them with MPTEMP.)
    mapdl.mp("ALPX", mid, alpha)
    
    # Thermal conductivity (isotropic). If anisotropic, you can define KYY, KZZ as well.
    mapdl.mp("KXX", mid, k)
    
    # Specific heat capacity (mass-based, J/kg-K). 
    # If you have volumetric C, do (volumetric_C / dens).
    mapdl.mp("C", mid, cp)

print("All materials for thermo-mechanical analysis have been defined.")

# You can now proceed to assign materials to volumes (via 'vatt' or 'mat' commands),
# mesh, define boundary conditions (thermal and structural), and solve.


In [ ]:
#~~~~~~~~For Job Scheduling~~~~~~~~~~~~~#
import os
path=os.getcwd()
jname='user_jobname'
mapdl=launch_mapdl(run_location=path,override=True)
mapdl.ignore_errors=True

In [ ]:
#~~~~~~~~For Job Scheduling~~~~~~~~~~~~~#
import os
path=os.getcwd()
jname='user_jobname'
mapdl=launch_mapdl(run_location=path,override=True)
mapdl.ignore_errors=True

In [ ]:
#~~~~~~~~For Job Scheduling~~~~~~~~~~~~~#
import os
path=os.getcwd()
jname='user_jobname'
mapdl=launch_mapdl(run_location=path,override=True)
mapdl.ignore_errors=True

In [ ]:
#~~~~~~~~For Job Scheduling~~~~~~~~~~~~~#
import os
path=os.getcwd()
jname='user_jobname'
mapdl=launch_mapdl(run_location=path,override=True)
mapdl.ignore_errors=True

In [ ]:
#~~~~~~~~For Job Scheduling~~~~~~~~~~~~~#
import os
path=os.getcwd()
jname='user_jobname'
mapdl=launch_mapdl(run_location=path,override=True)
mapdl.ignore_errors=True

In [ ]:
#~~~~~~~~For Job Scheduling~~~~~~~~~~~~~#
import os
path=os.getcwd()
jname='user_jobname'
mapdl=launch_mapdl(run_location=path,override=True)
mapdl.ignore_errors=True

In [ ]:
# 5) (Optional) Define element types, mesh, etc.
##############################################################################
# For example:
mapdl.et(1, "SOLID185")   # 3D structural brick
mapdl.vmesh("ALL")        # mesh all volumes

# Ready for loads, boundary conditions, solve, etc.
mapdl.finish()
mapdl.run("/SOLU")
# e.g. apply constraints, loads, then:
# mapdl.solve()
# mapdl.finish()

##############################################################################
# 6) End or keep interactive
##############################################################################
print("Geometry creation done. You can now inspect the volumes in MAPDL.")
# mapdl.exit()  # close if desired


Geometry Construction and Material Defination

In [ ]:
mapdl.exit()

In [ ]:
from ansys.mapdl.core import launch_mapdl
import math
import sys

# Launch a MAPDL instance (adjust parameters as needed)
mapdl = launch_mapdl()

##############################################################################
# 1) Define your geometry data (same as your SpaceClaim script)
##############################################################################
layers = [
    {"name": "Substrate",    "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "C4_Bumps",     "length": 30.0, "width": 30.0, "thickness": 0.1},
    {"name": "Interposer",   "length": 25.0, "width": 25.0, "thickness": 0.3},
    {"name": "MicroBumps",   "length": 25.0, "width": 25.0, "thickness": 0.05},
    {"name": "TIM_pads",     "length": 30.0, "width": 30.0, "thickness": 0.05},
    {"name": "HeatSpreader", "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "TIM2_pads",    "length": 35.0, "width": 35.0, "thickness": 0.05},
    # Instead of a single 2 mm block, we will build a heatsink with fins.
    {"name": "HeatSink",     "length": 40.0, "width": 40.0, "thickness": 2.0},
]

chiplets = [
    {"name": "Chiplet_1", "x_offset": 5.0,  "y_offset": 5.0,  "length": 5.0, "width": 5.0, "thickness": 0.2},
    {"name": "Chiplet_2", "x_offset": 10.0, "y_offset": 5.0,  "length": 4.0, "width": 4.0, "thickness": 0.2},
    {"name": "Chiplet_3", "x_offset": 15.0, "y_offset": 5.0,  "length": 6.0, "width": 6.0, "thickness": 0.2},
]

max_length = max(layer["length"] for layer in layers)
max_width  = max(layer["width"]  for layer in layers)

# Ensure we are in the preprocessor (PREP7) before geometry creation
mapdl.prep7()

##############################################################################
# 2) Create the geometry layer-by-layer
##############################################################################
current_z = 0.0

for i, layer in enumerate(layers):
    layer_name = layer["name"]
    L_mm = layer["length"]
    W_mm = layer["width"]
    T_mm = layer["thickness"]

    # Center each layer within the maximum extents
    x1 = (max_length - L_mm) / 2.0
    y1 = (max_width - W_mm) / 2.0
    x2 = x1 + L_mm
    y2 = y1 + W_mm
    z1 = current_z
    z2 = current_z + T_mm

    print(f"Creating layer {i+1}: {layer_name}")
    print(f"Coordinates: x1={x1}, x2={x2}, y1={y1}, y2={y2}, z1={z1}, z2={z2}")

    if layer_name == "HeatSink":
        # Create the base of the heatsink
        base_thickness_mm = 1.0
        fin_height_mm = 6.0
        mapdl.block(x1, x2, y1, y2, z1, z1 + base_thickness_mm)
        mapdl.run("! Created HeatSink_Base volume")

        # Create fins for the heatsink
        fin_count = 13
        fin_thickness = 1.0
        fin_gap = 2.0
        margin_x = 2.0
        fin_start_x = x1 + margin_x
        current_x = fin_start_x

        for fin_index in range(fin_count):
            if current_x + fin_thickness > x2 - margin_x:
                break
            fin_x1 = current_x
            fin_x2 = current_x + fin_thickness
            fin_y1 = y1
            fin_y2 = y2
            fin_z1 = z1 + base_thickness_mm
            fin_z2 = z1 + base_thickness_mm + fin_height_mm
            mapdl.block(fin_x1, fin_x2, fin_y1, fin_y2, fin_z1, fin_z2)
            mapdl.run("! Created HeatSink_Fin #{}".format(fin_index + 1))
            current_x += (fin_thickness + fin_gap)

        current_z += T_mm

    elif layer_name == "MicroBumps":
        # Create the MicroBumps layer
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created MicroBumps layer")
        current_z += T_mm

        # Create chiplets on top of the MicroBumps layer
        max_chip_thickness_mm = 0.0
        for chip in chiplets:
            chip_name = chip["name"]
            c_offset_x = chip["x_offset"]
            c_offset_y = chip["y_offset"]
            c_length_mm = chip["length"]
            c_width_mm = chip["width"]
            c_thk_mm = chip["thickness"]
            chip_x1 = x1 + c_offset_x
            chip_y1 = y1 + c_offset_y
            chip_x2 = chip_x1 + c_length_mm
            chip_y2 = chip_y1 + c_width_mm
            chip_z1 = current_z
            chip_z2 = chip_z1 + c_thk_mm
            mapdl.block(chip_x1, chip_x2, chip_y1, chip_y2, chip_z1, chip_z2)
            mapdl.run("! Created chip: '{}'".format(chip_name))
            if c_thk_mm > max_chip_thickness_mm:
                max_chip_thickness_mm = c_thk_mm

        # Advance stacking height by the tallest chip
        current_z += max_chip_thickness_mm

    else:
        # Create all other layers as simple blocks
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created layer volume for '{}'".format(layer_name))
        current_z += T_mm

# Plot the geometry (this may change the current MAPDL mode)
#mapdl.vplot()
print("Geometry creation done. You can now inspect the volumes in MAPDL.")

# Re-enter preprocessor mode to continue (important for subsequent commands)
mapdl.prep7()

##############################################################################
# 3) Material Property Definitions for thermo-mechanical analysis
##############################################################################
material_data = {
    "Substrate": {
        "mat_id": 1,
        "E": 3.0e10,
        "nu": 0.35,
        "dens": 2.5e3,
        "alpha": 1.0e-5,
        "kxx": 10.0,
        "cp": 800.0,
    },
    "C4_Bumps": {
        "mat_id": 2,
        "E": 5.0e10,
        "nu": 0.33,
        "dens": 8.9e3,
        "alpha": 2.3e-5,
        "kxx": 40.0,
        "cp": 200.0,
    },
    "Interposer": {
        "mat_id": 3,
        "E": 2.5e10,
        "nu": 0.34,
        "dens": 2.7e3,
        "alpha": 1.2e-5,
        "kxx": 20.0,
        "cp": 700.0,
    },
    "MicroBumps": {
        "mat_id": 4,
        "E": 2.0e10,
        "nu": 0.32,
        "dens": 2.7e3,
        "alpha": 2.2e-5,
        "kxx": 25.0,
        "cp": 210.0,
    },
    "TIM_pads": {
        "mat_id": 5,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSpreader": {
        "mat_id": 6,
        "E": 7.0e10,
        "nu": 0.29,
        "dens": 8.9e3,
        "alpha": 1.7e-5,
        "kxx": 400.0,
        "cp": 385.0,
    },
    "TIM2_pads": {
        "mat_id": 7,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSink": {
        "mat_id": 8,
        "E": 6.9e10,
        "nu": 0.33,
        "dens": 2.7e3,
        "alpha": 2.4e-5,
        "kxx": 205.0,
        "cp": 900.0,
    },
    "Chiplet": {
        "mat_id": 9,
        "E": 1.0e11,
        "nu": 0.28,
        "dens": 2.33e3,
        "alpha": 2.6e-6,
        "kxx": 149.0,
        "cp": 700.0,
    },
}

# Loop through the dictionary and define each material in MAPDL
for mat_name, props in material_data.items():
    mid   = props["mat_id"]
    E     = props["E"]
    nu    = props["nu"]
    dens  = props["dens"]
    alpha = props["alpha"]
    k     = props["kxx"]
    cp    = props["cp"]

    # Reset the temperature table and define a reference temperature
    mapdl.mptemp("", 0)
    mapdl.mptemp(1, 25)
    # Define material properties
    mapdl.mp("EX",   mid, E)
    mapdl.mp("PRXY", mid, nu)
    mapdl.mp("DENS", mid, dens)
    mapdl.mp("ALPX", mid, alpha)
    mapdl.mp("KXX", mid, k)
    mapdl.mp("C", mid, cp)

print("All materials for thermo-mechanical analysis have been defined.")

# Continue with meshing, boundary conditions, and solution setup as needed.


In [ ]:
mapdl.exit()

In [ ]:
from ansys.mapdl.core import launch_mapdl
import math

#------------------------------------------------------------------------------
# 1) Launch MAPDL and define geometry data
#------------------------------------------------------------------------------
mapdl = launch_mapdl()  # Launch MAPDL instance (adjust launch options as needed)

# Geometry layer data (as defined in your SpaceClaim script)
layers = [
    {"name": "Substrate",    "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "C4_Bumps",     "length": 30.0, "width": 30.0, "thickness": 0.1},
    {"name": "Interposer",   "length": 25.0, "width": 25.0, "thickness": 0.3},
    {"name": "MicroBumps",   "length": 25.0, "width": 25.0, "thickness": 0.05},
    {"name": "TIM_pads",     "length": 30.0, "width": 30.0, "thickness": 0.05},
    {"name": "HeatSpreader", "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "TIM2_pads",    "length": 35.0, "width": 35.0, "thickness": 0.05},
    # Instead of a single 2 mm block, we will build a heatsink with fins.
    {"name": "HeatSink",     "length": 40.0, "width": 40.0, "thickness": 2.0},
]

chiplets = [
    {"name": "Chiplet_1", "x_offset": 5.0,  "y_offset": 5.0,  "length": 5.0, "width": 5.0, "thickness": 0.2},
    {"name": "Chiplet_2", "x_offset": 10.0, "y_offset": 5.0,  "length": 4.0, "width": 4.0, "thickness": 0.2},
    {"name": "Chiplet_3", "x_offset": 15.0, "y_offset": 5.0,  "length": 6.0, "width": 6.0, "thickness": 0.2},
]

max_length = max(layer["length"] for layer in layers)
max_width  = max(layer["width"]  for layer in layers)

# Always start in the preprocessor
mapdl.prep7()

#------------------------------------------------------------------------------
# 2) Create the geometry layer-by-layer
#------------------------------------------------------------------------------
current_z = 0.0

for i, layer in enumerate(layers):
    layer_name = layer["name"]
    L_mm = layer["length"]
    W_mm = layer["width"]
    T_mm = layer["thickness"]

    # Center each layer within the maximum extents
    x1 = (max_length - L_mm) / 2.0
    y1 = (max_width  - W_mm) / 2.0
    x2 = x1 + L_mm
    y2 = y1 + W_mm
    z1 = current_z
    z2 = current_z + T_mm

    print(f"Creating layer {i+1}: {layer_name}")
    print(f"Coordinates: x1={x1}, x2={x2}, y1={y1}, y2={y2}, z1={z1}, z2={z2}")

    if layer_name == "HeatSink":
        # Create heatsink base and then fins
        base_thickness_mm = 1.0
        fin_height_mm = 6.0
        mapdl.block(x1, x2, y1, y2, z1, z1 + base_thickness_mm)
        mapdl.run("! Created HeatSink_Base volume")

        # Create fins for the heatsink
        fin_count = 13
        fin_thickness = 1.0
        fin_gap = 2.0
        margin_x = 2.0
        current_x = x1 + margin_x

        for fin_index in range(fin_count):
            # Ensure we remain within the base boundaries
            if current_x + fin_thickness > x2 - margin_x:
                break
            fin_x1 = current_x
            fin_x2 = current_x + fin_thickness
            fin_y1 = y1
            fin_y2 = y2
            fin_z1 = z1 + base_thickness_mm
            fin_z2 = fin_z1 + fin_height_mm
            mapdl.block(fin_x1, fin_x2, fin_y1, fin_y2, fin_z1, fin_z2)
            mapdl.run("! Created HeatSink_Fin #{}".format(fin_index + 1))
            current_x += fin_thickness + fin_gap

        current_z += T_mm

    elif layer_name == "MicroBumps":
        # Create the MicroBumps layer
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created MicroBumps layer")
        current_z += T_mm

        # Place chiplets on top of the MicroBumps layer
        max_chip_thickness_mm = 0.0
        for chip in chiplets:
            chip_name = chip["name"]
            c_offset_x = chip["x_offset"]
            c_offset_y = chip["y_offset"]
            c_length_mm = chip["length"]
            c_width_mm = chip["width"]
            c_thk_mm = chip["thickness"]
            chip_x1 = x1 + c_offset_x
            chip_y1 = y1 + c_offset_y
            chip_x2 = chip_x1 + c_length_mm
            chip_y2 = chip_y1 + c_width_mm
            chip_z1 = current_z
            chip_z2 = chip_z1 + c_thk_mm
            mapdl.block(chip_x1, chip_x2, chip_y1, chip_y2, chip_z1, chip_z2)
            mapdl.run("! Created chip: '{}'".format(chip_name))
            if c_thk_mm > max_chip_thickness_mm:
                max_chip_thickness_mm = c_thk_mm

        current_z += max_chip_thickness_mm

    else:
        # Create any other layer as a simple block
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created layer volume for '{}'".format(layer_name))
        current_z += T_mm

#------------------------------------------------------------------------------
# Optional: Plot the geometry
#------------------------------------------------------------------------------
# NOTE: In some cases the interactive plot window can cause the MAPDL connection
# to be lost if it is closed. To avoid this, we use wait=False so that the
# command returns immediately. If you encounter issues, you may comment this out.
try:
    mapdl.vplot(wait=False)
except Exception as e:
    print("Warning: vplot failed (this may be intentional in your environment):", e)

# IMPORTANT: Do not close the MAPDL connection by closing the plot window if you
# intend to continue running further commands.

#------------------------------------------------------------------------------
# 3) Re-enter PREP7 and define material properties
#------------------------------------------------------------------------------
mapdl.prep7()  # Ensure we are in preprocessor mode

material_data = {
    "Substrate": {
        "mat_id": 1,
        "E": 3.0e10,
        "nu": 0.35,
        "dens": 2.5e3,
        "alpha": 1.0e-5,
        "kxx": 10.0,
        "cp": 800.0,
    },
    "C4_Bumps": {
        "mat_id": 2,
        "E": 5.0e10,
        "nu": 0.33,
        "dens": 8.9e3,
        "alpha": 2.3e-5,
        "kxx": 40.0,
        "cp": 200.0,
    },
    "Interposer": {
        "mat_id": 3,
        "E": 2.5e10,
        "nu": 0.34,
        "dens": 2.7e3,
        "alpha": 1.2e-5,
        "kxx": 20.0,
        "cp": 700.0,
    },
    "MicroBumps": {
        "mat_id": 4,
        "E": 2.0e10,
        "nu": 0.32,
        "dens": 2.7e3,
        "alpha": 2.2e-5,
        "kxx": 25.0,
        "cp": 210.0,
    },
    "TIM_pads": {
        "mat_id": 5,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSpreader": {
        "mat_id": 6,
        "E": 7.0e10,
        "nu": 0.29,
        "dens": 8.9e3,
        "alpha": 1.7e-5,
        "kxx": 400.0,
        "cp": 385.0,
    },
    "TIM2_pads": {
        "mat_id": 7,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSink": {
        "mat_id": 8,
        "E": 6.9e10,
        "nu": 0.33,
        "dens": 2.7e3,
        "alpha": 2.4e-5,
        "kxx": 205.0,
        "cp": 900.0,
    },
    "Chiplet": {
        "mat_id": 9,
        "E": 1.0e11,
        "nu": 0.28,
        "dens": 2.33e3,
        "alpha": 2.6e-6,
        "kxx": 149.0,
        "cp": 700.0,
    },
}

for mat_name, props in material_data.items():
    mid   = props["mat_id"]
    E     = props["E"]
    nu    = props["nu"]
    dens  = props["dens"]
    alpha = props["alpha"]
    k     = props["kxx"]
    cp    = props["cp"]

    # Reset the temperature table and define a reference temperature
    mapdl.mptemp("", 0)
    mapdl.mptemp(1, 25)
    # Define material properties in MAPDL
    mapdl.mp("EX",   mid, E)
    mapdl.mp("PRXY", mid, nu)
    mapdl.mp("DENS", mid, dens)
    mapdl.mp("ALPX", mid, alpha)
    mapdl.mp("KXX", mid, k)
    mapdl.mp("C", mid, cp)

print("All materials for thermo-mechanical analysis have been defined.")

#------------------------------------------------------------------------------
# Continue with meshing, boundary conditions, and solution setup as needed.
#------------------------------------------------------------------------------


In [ ]:
from ansys.mapdl.core import launch_mapdl
import math

#------------------------------------------------------------------------------
# 1) Launch MAPDL and define geometry data
#------------------------------------------------------------------------------
mapdl = launch_mapdl()  # Launch MAPDL instance (adjust options if needed)

# Define geometry layer data (same as your SpaceClaim script)
layers = [
    {"name": "Substrate",    "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "C4_Bumps",     "length": 30.0, "width": 30.0, "thickness": 0.1},
    {"name": "Interposer",   "length": 25.0, "width": 25.0, "thickness": 0.3},
    {"name": "MicroBumps",   "length": 25.0, "width": 25.0, "thickness": 0.05},
    {"name": "TIM_pads",     "length": 30.0, "width": 30.0, "thickness": 0.05},
    {"name": "HeatSpreader", "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "TIM2_pads",    "length": 35.0, "width": 35.0, "thickness": 0.05},
    # Instead of a single 2 mm block, we build a heatsink with fins.
    {"name": "HeatSink",     "length": 40.0, "width": 40.0, "thickness": 2.0},
]

chiplets = [
    {"name": "Chiplet_1", "x_offset": 5.0,  "y_offset": 5.0,  "length": 5.0, "width": 5.0, "thickness": 0.2},
    {"name": "Chiplet_2", "x_offset": 10.0, "y_offset": 5.0,  "length": 4.0, "width": 4.0, "thickness": 0.2},
    {"name": "Chiplet_3", "x_offset": 15.0, "y_offset": 5.0,  "length": 6.0, "width": 6.0, "thickness": 0.2},
]

max_length = max(layer["length"] for layer in layers)
max_width  = max(layer["width"]  for layer in layers)

# Always start in PREP7
mapdl.prep7()

#------------------------------------------------------------------------------
# 2) Create the geometry layer-by-layer
#------------------------------------------------------------------------------
current_z = 0.0

for i, layer in enumerate(layers):
    layer_name = layer["name"]
    L_mm = layer["length"]
    W_mm = layer["width"]
    T_mm = layer["thickness"]

    # Center the layer within the overall extents
    x1 = (max_length - L_mm) / 2.0
    y1 = (max_width  - W_mm) / 2.0
    x2 = x1 + L_mm
    y2 = y1 + W_mm
    z1 = current_z
    z2 = current_z + T_mm

    print(f"Creating layer {i+1}: {layer_name}")
    print(f"Coordinates: x1={x1}, x2={x2}, y1={y1}, y2={y2}, z1={z1}, z2={z2}")

    if layer_name == "HeatSink":
        # Create heatsink base and fins
        base_thickness_mm = 1.0
        fin_height_mm = 6.0

        # Create the heatsink base
        mapdl.block(x1, x2, y1, y2, z1, z1 + base_thickness_mm)
        mapdl.run("! Created HeatSink_Base volume")

        # Create fins
        fin_count = 13
        fin_thickness = 1.0
        fin_gap = 2.0
        margin_x = 2.0
        current_x = x1 + margin_x

        for fin_index in range(fin_count):
            if current_x + fin_thickness > x2 - margin_x:
                break
            fin_x1 = current_x
            fin_x2 = current_x + fin_thickness
            fin_y1 = y1
            fin_y2 = y2
            fin_z1 = z1 + base_thickness_mm
            fin_z2 = fin_z1 + fin_height_mm
            mapdl.block(fin_x1, fin_x2, fin_y1, fin_y2, fin_z1, fin_z2)
            mapdl.run("! Created HeatSink_Fin #{}".format(fin_index + 1))
            current_x += fin_thickness + fin_gap

        current_z += T_mm

    elif layer_name == "MicroBumps":
        # Create the MicroBumps layer
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created MicroBumps layer")
        current_z += T_mm

        # Create chiplets on top of the MicroBumps layer
        max_chip_thickness_mm = 0.0
        for chip in chiplets:
            chip_name = chip["name"]
            c_offset_x = chip["x_offset"]
            c_offset_y = chip["y_offset"]
            c_length_mm = chip["length"]
            c_width_mm = chip["width"]
            c_thk_mm = chip["thickness"]
            chip_x1 = x1 + c_offset_x
            chip_y1 = y1 + c_offset_y
            chip_x2 = chip_x1 + c_length_mm
            chip_y2 = chip_y1 + c_width_mm
            chip_z1 = current_z
            chip_z2 = chip_z1 + c_thk_mm
            mapdl.block(chip_x1, chip_x2, chip_y1, chip_y2, chip_z1, chip_z2)
            mapdl.run("! Created chip: '{}'".format(chip_name))
            if c_thk_mm > max_chip_thickness_mm:
                max_chip_thickness_mm = c_thk_mm

        current_z += max_chip_thickness_mm

    else:
        # Create other layers as simple blocks
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created layer volume for '{}'".format(layer_name))
        current_z += T_mm

#------------------------------------------------------------------------------
# Instead of using vplot (which may cause the MAPDL session to exit if closed),
# we directly define a global element size for meshing.
#------------------------------------------------------------------------------
mapdl.lesize("ALL", 0.15, layer1=1)

#------------------------------------------------------------------------------
# 3) Re-enter PREP7 and define material properties
#------------------------------------------------------------------------------
mapdl.prep7()  # Ensure we are in the preprocessor

material_data = {
    "Substrate": {
        "mat_id": 1,
        "E": 3.0e10,
        "nu": 0.35,
        "dens": 2.5e3,
        "alpha": 1.0e-5,
        "kxx": 10.0,
        "cp": 800.0,
    },
    "C4_Bumps": {
        "mat_id": 2,
        "E": 5.0e10,
        "nu": 0.33,
        "dens": 8.9e3,
        "alpha": 2.3e-5,
        "kxx": 40.0,
        "cp": 200.0,
    },
    "Interposer": {
        "mat_id": 3,
        "E": 2.5e10,
        "nu": 0.34,
        "dens": 2.7e3,
        "alpha": 1.2e-5,
        "kxx": 20.0,
        "cp": 700.0,
    },
    "MicroBumps": {
        "mat_id": 4,
        "E": 2.0e10,
        "nu": 0.32,
        "dens": 2.7e3,
        "alpha": 2.2e-5,
        "kxx": 25.0,
        "cp": 210.0,
    },
    "TIM_pads": {
        "mat_id": 5,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSpreader": {
        "mat_id": 6,
        "E": 7.0e10,
        "nu": 0.29,
        "dens": 8.9e3,
        "alpha": 1.7e-5,
        "kxx": 400.0,
        "cp": 385.0,
    },
    "TIM2_pads": {
        "mat_id": 7,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSink": {
        "mat_id": 8,
        "E": 6.9e10,
        "nu": 0.33,
        "dens": 2.7e3,
        "alpha": 2.4e-5,
        "kxx": 205.0,
        "cp": 900.0,
    },
    "Chiplet": {
        "mat_id": 9,
        "E": 1.0e11,
        "nu": 0.28,
        "dens": 2.33e3,
        "alpha": 2.6e-6,
        "kxx": 149.0,
        "cp": 700.0,
    },
}

# Loop over each material and define its properties in MAPDL
for mat_name, props in material_data.items():
    mid   = props["mat_id"]
    E     = props["E"]
    nu    = props["nu"]
    dens  = props["dens"]
    alpha = props["alpha"]
    k     = props["kxx"]
    cp    = props["cp"]

    # Reset the temperature table and set a reference temperature (25°C)
    #mapdl.mptemp("", 0)
    mapdl.mptemp(1, 25)

    # Define material properties
    mapdl.mp("EX",   mid, E)
    mapdl.mp("PRXY", mid, nu)
    mapdl.mp("DENS", mid, dens)
    mapdl.mp("ALPX", mid, alpha)
    mapdl.mp("KXX", mid, k)
    mapdl.mp("C", mid, cp)

print("All materials for thermo-mechanical analysis have been defined.")

In [ ]:
from ansys.mapdl.core import launch_mapdl
import math

# Launch MAPDL and enter the preprocessor
mapdl = launch_mapdl()
mapdl.prep7()

#------------------------------------------------------------------------------
# 1) Define geometry data
#------------------------------------------------------------------------------
layers = [
    {"name": "Substrate",    "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "C4_Bumps",     "length": 30.0, "width": 30.0, "thickness": 0.1},
    {"name": "Interposer",   "length": 25.0, "width": 25.0, "thickness": 0.3},
    {"name": "MicroBumps",   "length": 25.0, "width": 25.0, "thickness": 0.05},
    {"name": "TIM_pads",     "length": 30.0, "width": 30.0, "thickness": 0.05},
    {"name": "HeatSpreader", "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "TIM2_pads",    "length": 35.0, "width": 35.0, "thickness": 0.05},
    # The heatsink will be built with a base and multiple fins.
    {"name": "HeatSink",     "length": 40.0, "width": 40.0, "thickness": 2.0},
]

chiplets = [
    {"name": "Chiplet_1", "x_offset": 5.0,  "y_offset": 5.0,  "length": 5.0, "width": 5.0, "thickness": 0.2},
    {"name": "Chiplet_2", "x_offset": 10.0, "y_offset": 5.0,  "length": 4.0, "width": 4.0, "thickness": 0.2},
    {"name": "Chiplet_3", "x_offset": 15.0, "y_offset": 5.0,  "length": 6.0, "width": 6.0, "thickness": 0.2},
]

max_length = max(layer["length"] for layer in layers)
max_width  = max(layer["width"] for layer in layers)

#------------------------------------------------------------------------------
# 2) Create the geometry layer-by-layer with named selections
#------------------------------------------------------------------------------
current_z = 0.0

for i, layer in enumerate(layers):
    layer_name = layer["name"]
    L_mm = layer["length"]
    W_mm = layer["width"]
    T_mm = layer["thickness"]

    # Center each layer within the overall extents
    x1 = (max_length - L_mm) / 2.0
    y1 = (max_width  - W_mm) / 2.0
    x2 = x1 + L_mm
    y2 = y1 + W_mm
    z1 = current_z
    z2 = current_z + T_mm

    print(f"Creating layer {i+1}: {layer_name}")
    print(f"Coordinates: x1={x1}, x2={x2}, y1={y1}, y2={y2}, z1={z1}, z2={z2}")

    if layer_name == "HeatSink":
        # For the heatsink, create a base and fins, then assign them all the same component name.
        base_thickness = 1.0
        fin_height = 6.0

        # Create the heatsink base
        mapdl.block(x1, x2, y1, y2, z1, z1 + base_thickness)
        mapdl.run("! Created HeatSink base volume")
        # Assign the component name "HeatSink" to the most recently created volume
        mapdl.vatt("LAST", "comp", "HeatSink")

        # Create fins for the heatsink
        fin_count = 13
        fin_thickness = 1.0
        fin_gap = 2.0
        margin_x = 2.0
        current_x = x1 + margin_x

        for fin_index in range(fin_count):
            if current_x + fin_thickness > x2 - margin_x:
                break
            fin_x1 = current_x
            fin_x2 = current_x + fin_thickness
            fin_y1 = y1
            fin_y2 = y2
            fin_z1 = z1 + base_thickness
            fin_z2 = fin_z1 + fin_height
            mapdl.block(fin_x1, fin_x2, fin_y1, fin_y2, fin_z1, fin_z2)
            mapdl.run("! Created HeatSink fin #{}".format(fin_index + 1))
            # Also assign "HeatSink" as the component name for each fin volume
            mapdl.vatt("LAST", "comp", "HeatSink")
            current_x += fin_thickness + fin_gap

        current_z += T_mm

    elif layer_name == "MicroBumps":
        # Create the MicroBumps base volume and assign its component name
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created MicroBumps layer volume")
        mapdl.vatt("LAST", "comp", "MicroBumps")
        current_z += T_mm

        # Now create chiplets on top of the MicroBumps layer; each chiplet gets its own name.
        max_chip_thickness = 0.0
        for chip in chiplets:
            chip_name = chip["name"]
            c_offset_x = chip["x_offset"]
            c_offset_y = chip["y_offset"]
            c_length = chip["length"]
            c_width = chip["width"]
            c_thickness = chip["thickness"]
            chip_x1 = x1 + c_offset_x
            chip_y1 = y1 + c_offset_y
            chip_x2 = chip_x1 + c_length
            chip_y2 = chip_y1 + c_width
            chip_z1 = current_z
            chip_z2 = chip_z1 + c_thickness
            mapdl.block(chip_x1, chip_x2, chip_y1, chip_y2, chip_z1, chip_z2)
            mapdl.run("! Created chiplet: '{}'".format(chip_name))
            # Assign the chiplet's unique name as its component
            mapdl.vatt("LAST", "comp", chip_name)
            if c_thickness > max_chip_thickness:
                max_chip_thickness = c_thickness
        current_z += max_chip_thickness

    else:
        # For all other layers, create a volume and assign a component name equal to the layer name.
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created {} volume".format(layer_name))
        mapdl.vatt("LAST", "comp", layer_name)
        current_z += T_mm

print("Geometry creation complete with named selections.")
# Ensure we are in the preprocessor
#mapdl.prep7()

##############################################################################
# 3) Define Material Properties 
##############################################################################
mapdl.emunit(lab="MKS")
material_data = {
    "Substrate": {
        "mat_id": 1,
        "E": 3.0e10,
        "nu": 0.35,
        "dens": 2.5e3,
        "alpha": 1.0e-5,
        "kxx": 10.0,
        "cp": 800.0,
    },
    "C4_Bumps": {
        "mat_id": 2,
        "E": 5.0e10,
        "nu": 0.33,
        "dens": 8.9e3,
        "alpha": 2.3e-5,
        "kxx": 40.0,
        "cp": 200.0,
    },
    "Interposer": {
        "mat_id": 3,
        "E": 2.5e10,
        "nu": 0.34,
        "dens": 2.7e3,
        "alpha": 1.2e-5,
        "kxx": 20.0,
        "cp": 700.0,
    },
    "MicroBumps": {
        "mat_id": 4,
        "E": 2.0e10,
        "nu": 0.32,
        "dens": 2.7e3,
        "alpha": 2.2e-5,
        "kxx": 25.0,
        "cp": 210.0,
    },
    "TIM_pads": {
        "mat_id": 5,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSpreader": {
        "mat_id": 6,
        "E": 7.0e10,
        "nu": 0.29,
        "dens": 8.9e3,
        "alpha": 1.7e-5,
        "kxx": 400.0,
        "cp": 385.0,
    },
    "TIM2_pads": {
        "mat_id": 7,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSink": {
        "mat_id": 8,
        "E": 6.9e10,
        "nu": 0.33,
        "dens": 2.7e3,
        "alpha": 2.4e-5,
        "kxx": 205.0,
        "cp": 900.0,
    },
    "Chiplet": {
        "mat_id": 9,
        "E": 1.0e11,
        "nu": 0.28,
        "dens": 2.33e3,
        "alpha": 2.6e-6,
        "kxx": 149.0,
        "cp": 700.0,
    },
}

# Loop over the dictionary to define each material in MAPDL
for mat_name, props in material_data.items():
    mid   = props["mat_id"]
    E     = props["E"]
    nu    = props["nu"]
    dens  = props["dens"]
    alpha = props["alpha"]
    k     = props["kxx"]
    cp    = props["cp"]

    # Reset the temperature table and set a reference temperature (e.g., 25°C)
    #mapdl.mptemp("", 0)
    #mapdl.mptemp(1, 25)

    # Define material properties
    mapdl.mp("EX",   mid, E)
    mapdl.mp("PRXY", mid, nu)
    mapdl.mp("DENS", mid, dens)
    mapdl.mp("ALPX", mid, alpha)
    mapdl.mp("KXX", mid, k)
    mapdl.mp("C", mid, cp)

print("Materials have been defined in MAPDL.")

##############################################################################
# 4) Assign Materials to Volumes via Named Selections
##############################################################################

# Always select all entities before starting assignments
mapdl.allsel()

# Substrate (component name "Substrate")
mapdl.vsel("S", "comp", "Substrate")
mapdl.mat(material_data["Substrate"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# C4_Bumps (component name "C4_Bumps")
mapdl.vsel("S", "comp", "C4_Bumps")
mapdl.mat(material_data["C4_Bumps"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# Interposer (component name "Interposer")
mapdl.vsel("S", "comp", "Interposer")
mapdl.mat(material_data["Interposer"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# MicroBumps (component name "MicroBumps")
mapdl.vsel("S", "comp", "MicroBumps")
mapdl.mat(material_data["MicroBumps"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# TIM_pads (component name "TIM_pads")
mapdl.vsel("S", "comp", "TIM_pads")
mapdl.mat(material_data["TIM_pads"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# HeatSpreader (component name "HeatSpreader")
mapdl.vsel("S", "comp", "HeatSpreader")
mapdl.mat(material_data["HeatSpreader"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# TIM2_pads (component name "TIM2_pads")
mapdl.vsel("S", "comp", "TIM2_pads")
mapdl.mat(material_data["TIM2_pads"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# HeatSink (component name "HeatSink" covers the base and fins)
mapdl.vsel("S", "comp", "HeatSink")
mapdl.mat(material_data["HeatSink"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# Chiplets (each chiplet has a unique component name)
for chip in ["Chiplet_1", "Chiplet_2", "Chiplet_3"]:
    mapdl.vsel("S", "comp", chip)
    mapdl.mat(material_data["Chiplet"]["mat_id"])
    mapdl.vsll("ALL")
    mapdl.allsel()

print("Material assignment to named selections is complete.")

In [ ]:
mapdl.cmlist()

In [ ]:
# Ensure we are in the preprocessor
mapdl.prep7()

##############################################################################
# 1) Define Material Properties 
##############################################################################
material_data = {
    "Substrate": {
        "mat_id": 1,
        "E": 3.0e10,
        "nu": 0.35,
        "dens": 2.5e3,
        "alpha": 1.0e-5,
        "kxx": 10.0,
        "cp": 800.0,
    },
    "C4_Bumps": {
        "mat_id": 2,
        "E": 5.0e10,
        "nu": 0.33,
        "dens": 8.9e3,
        "alpha": 2.3e-5,
        "kxx": 40.0,
        "cp": 200.0,
    },
    "Interposer": {
        "mat_id": 3,
        "E": 2.5e10,
        "nu": 0.34,
        "dens": 2.7e3,
        "alpha": 1.2e-5,
        "kxx": 20.0,
        "cp": 700.0,
    },
    "MicroBumps": {
        "mat_id": 4,
        "E": 2.0e10,
        "nu": 0.32,
        "dens": 2.7e3,
        "alpha": 2.2e-5,
        "kxx": 25.0,
        "cp": 210.0,
    },
    "TIM_pads": {
        "mat_id": 5,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSpreader": {
        "mat_id": 6,
        "E": 7.0e10,
        "nu": 0.29,
        "dens": 8.9e3,
        "alpha": 1.7e-5,
        "kxx": 400.0,
        "cp": 385.0,
    },
    "TIM2_pads": {
        "mat_id": 7,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSink": {
        "mat_id": 8,
        "E": 6.9e10,
        "nu": 0.33,
        "dens": 2.7e3,
        "alpha": 2.4e-5,
        "kxx": 205.0,
        "cp": 900.0,
    },
    "Chiplet": {
        "mat_id": 9,
        "E": 1.0e11,
        "nu": 0.28,
        "dens": 2.33e3,
        "alpha": 2.6e-6,
        "kxx": 149.0,
        "cp": 700.0,
    },
}

# Loop over the dictionary to define each material in MAPDL
for mat_name, props in material_data.items():
    mid   = props["mat_id"]
    E     = props["E"]
    nu    = props["nu"]
    dens  = props["dens"]
    alpha = props["alpha"]
    k     = props["kxx"]
    cp    = props["cp"]

    # Reset the temperature table and set a reference temperature (e.g., 25°C)
    mapdl.mptemp("", 0)
    mapdl.mptemp(1, 25)

    # Define material properties
    mapdl.mp("EX",   mid, E)
    mapdl.mp("PRXY", mid, nu)
    mapdl.mp("DENS", mid, dens)
    mapdl.mp("ALPX", mid, alpha)
    mapdl.mp("KXX", mid, k)
    mapdl.mp("C", mid, cp)

print("Materials have been defined in MAPDL.")

##############################################################################
# 2) Assign Materials to Volumes via Named Selections
##############################################################################

# Always select all entities before starting assignments
mapdl.allsel()

# Substrate (component name "Substrate")
mapdl.vsel("S", "comp", "Substrate")
mapdl.mat(material_data["Substrate"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# C4_Bumps (component name "C4_Bumps")
mapdl.vsel("S", "comp", "C4_Bumps")
mapdl.mat(material_data["C4_Bumps"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# Interposer (component name "Interposer")
mapdl.vsel("S", "comp", "Interposer")
mapdl.mat(material_data["Interposer"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# MicroBumps (component name "MicroBumps")
mapdl.vsel("S", "comp", "MicroBumps")
mapdl.mat(material_data["MicroBumps"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# TIM_pads (component name "TIM_pads")
mapdl.vsel("S", "comp", "TIM_pads")
mapdl.mat(material_data["TIM_pads"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# HeatSpreader (component name "HeatSpreader")
mapdl.vsel("S", "comp", "HeatSpreader")
mapdl.mat(material_data["HeatSpreader"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# TIM2_pads (component name "TIM2_pads")
mapdl.vsel("S", "comp", "TIM2_pads")
mapdl.mat(material_data["TIM2_pads"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# HeatSink (component name "HeatSink" covers the base and fins)
mapdl.vsel("S", "comp", "HeatSink")
mapdl.mat(material_data["HeatSink"]["mat_id"])
mapdl.vsll("ALL")
mapdl.allsel()

# Chiplets (each chiplet has a unique component name)
for chip in ["Chiplet_1", "Chiplet_2", "Chiplet_3"]:
    mapdl.vsel("S", "comp", chip)
    mapdl.mat(material_data["Chiplet"]["mat_id"])
    mapdl.vsll("ALL")
    mapdl.allsel()

print("Material assignment to named selections is complete.")


In [ ]:
from ansys.mapdl.core import launch_mapdl
import math

# Launch MAPDL and enter PREP7
mapdl = launch_mapdl()
mapdl.prep7()

#------------------------------------------------------------------------------
# 1) Define Geometry Data
#------------------------------------------------------------------------------
layers = [
    {"name": "Substrate",    "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "C4_Bumps",     "length": 30.0, "width": 30.0, "thickness": 0.1},
    {"name": "Interposer",   "length": 25.0, "width": 25.0, "thickness": 0.3},
    {"name": "MicroBumps",   "length": 25.0, "width": 25.0, "thickness": 0.05},
    {"name": "TIM_pads",     "length": 30.0, "width": 30.0, "thickness": 0.05},
    {"name": "HeatSpreader", "length": 35.0, "width": 35.0, "thickness": 0.5},
    {"name": "TIM2_pads",    "length": 35.0, "width": 35.0, "thickness": 0.05},
    # The HeatSink layer (to be built as an assembly of base and fins)
    {"name": "HeatSink",     "length": 40.0, "width": 40.0, "thickness": 2.0},
]

chiplets = [
    {"name": "Chiplet_1", "x_offset": 5.0,  "y_offset": 5.0,  "length": 5.0, "width": 5.0, "thickness": 0.2},
    {"name": "Chiplet_2", "x_offset": 10.0, "y_offset": 5.0,  "length": 4.0, "width": 4.0, "thickness": 0.2},
    {"name": "Chiplet_3", "x_offset": 15.0, "y_offset": 5.0,  "length": 6.0, "width": 6.0, "thickness": 0.2},
]

max_length = max(layer["length"] for layer in layers)
max_width  = max(layer["width"] for layer in layers)

#------------------------------------------------------------------------------
# 2) Create Geometry Layer-by-Layer and Define Components
#------------------------------------------------------------------------------
current_z = 0.0

for i, layer in enumerate(layers):
    layer_name = layer["name"]
    L_mm = layer["length"]
    W_mm = layer["width"]
    T_mm = layer["thickness"]

    # Center the layer within the overall extents
    x1 = (max_length - L_mm) / 2.0
    y1 = (max_width - W_mm) / 2.0
    x2 = x1 + L_mm
    y2 = y1 + W_mm
    z1 = current_z
    z2 = current_z + T_mm

    print(f"Creating layer {i+1}: {layer_name}")
    print(f"Coordinates: x1={x1}, x2={x2}, y1={y1}, y2={y2}, z1={z1}, z2={z2}")

    if layer_name == "HeatSink":
    # Create base
        base_thickness = 1.0
        fin_height = 6.0
        mapdl.block(x1, x2, y1, y2, z1, z1 + base_thickness)
        base_num = mapdl.geometry.vnum # store base volume number
        
        # Create fins
        fin_count = 13
        fin_thickness = 1.0
        fin_gap = 2.0
        margin_x = 2.0
        current_x = x1 + margin_x
        
        fin_nums = []  # store fin volume numbers
        
        for fin_index in range(fin_count):
            if current_x + fin_thickness > x2 - margin_x:
                break
                
            fin_x1 = current_x
            fin_x2 = current_x + fin_thickness
            fin_z1 = z1 + base_thickness
            fin_z2 = fin_z1 + fin_height
            
            mapdl.block(fin_x1, fin_x2, y1, y2, fin_z1, fin_z2)
            fin_nums.append(mapdl.geometry.vnum)
            current_x += fin_thickness + fin_gap
    
        # Select all volumes and create HeatSink component
        mapdl.vsel('S', 'VOLU', '', base_num)
        for fin_num in fin_nums:
            mapdl.vsel('A', 'VOLU', '', fin_num)
        mapdl.cm('HeatSink', 'VOLU')
    
        current_z += T_mm
    '''
    if layer_name == "HeatSink":
        # Create base
        base_thickness = 1.0
        fin_height = 6.0
        mapdl.block(x1, x2, y1, y2, z1, z1 + base_thickness)
        mapdl.cm("HeatSinkBase", "VOLU")
        
        # Create fins
        fin_count = 13
        fin_thickness = 1.0
        fin_gap = 2.0
        margin_x = 2.0
        current_x = x1 + margin_x
        
        # Initialize HeatSinkFins component
        mapdl.cm("HeatSinkFins", "VOLU")
        
        for fin_index in range(fin_count):
            if current_x + fin_thickness > x2 - margin_x:
                break
                
            fin_x1 = current_x
            fin_x2 = current_x + fin_thickness
            fin_z1 = z1 + base_thickness
            fin_z2 = fin_z1 + fin_height
            
            mapdl.block(fin_x1, fin_x2, y1, y2, fin_z1, fin_z2)
            mapdl.cmsel("S", "HeatSinkFins")
            mapdl.vsel("A", "LAST")
            mapdl.cm("HeatSinkFins", "VOLU")
            
            current_x += fin_thickness + fin_gap
        
        # Merge base and fins into HeatSink component
        mapdl.cmsel("S", "HeatSinkBase")
        mapdl.cmsel("A", "HeatSinkFins")
        mapdl.cm("HeatSink", "VOLU")

        
        current_z += T_mm
'''
    if layer_name == "MicroBumps":
        # Create the MicroBumps layer and assign its component
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run("! Created MicroBumps layer volume")
        mapdl.cm("MicroBumps", option="VOLU", item="LAST")
        current_z += T_mm

        # Create chiplets on top of the MicroBumps layer; each chiplet gets its own component.
        max_chip_thickness = 0.0
        for chip in chiplets:
            chip_name = chip["name"]
            c_offset_x = chip["x_offset"]
            c_offset_y = chip["y_offset"]
            c_length = chip["length"]
            c_width = chip["width"]
            c_thickness = chip["thickness"]
            chip_x1 = x1 + c_offset_x
            chip_y1 = y1 + c_offset_y
            chip_x2 = chip_x1 + c_length
            chip_y2 = chip_y1 + c_width
            chip_z1 = current_z
            chip_z2 = chip_z1 + c_thickness
            mapdl.block(chip_x1, chip_x2, chip_y1, chip_y2, chip_z1, chip_z2)
            mapdl.run(f"! Created chiplet: '{chip_name}'")
            # Create a component for this chiplet
            mapdl.cm(chip_name, option="VOLU", item="LAST")
            if c_thickness > max_chip_thickness:
                max_chip_thickness = c_thickness
        current_z += max_chip_thickness

    else:
        # For all other layers (Substrate, C4_Bumps, Interposer, TIM_pads, HeatSpreader, TIM2_pads)
        mapdl.block(x1, x2, y1, y2, z1, z2)
        mapdl.run(f"! Created {layer_name} volume")
        mapdl.cm(layer_name, option="VOLU", item="LAST")
        current_z += T_mm

print("Geometry creation complete and components have been defined.")

# Optional: Display a summary of the defined components.
print("Components defined:")
mapdl.allsel()
mapdl.cmlist()


In [ ]:
mapdl.ignore_errors = True

In [ ]:
# Ensure we're in PREP7
mapdl.prep7()

##############################################################################
# 1) Define Material Properties
##############################################################################
material_data = {
    "Substrate": {
        "mat_id": 1,
        "E": 3.0e10,
        "nu": 0.35,
        "dens": 2.5e3,
        "alpha": 1.0e-5,
        "kxx": 10.0,
        "cp": 800.0,
    },
    "C4_Bumps": {
        "mat_id": 2,
        "E": 5.0e10,
        "nu": 0.33,
        "dens": 8.9e3,
        "alpha": 2.3e-5,
        "kxx": 40.0,
        "cp": 200.0,
    },
    "Interposer": {
        "mat_id": 3,
        "E": 2.5e10,
        "nu": 0.34,
        "dens": 2.7e3,
        "alpha": 1.2e-5,
        "kxx": 20.0,
        "cp": 700.0,
    },
    "MicroBumps": {
        "mat_id": 4,
        "E": 2.0e10,
        "nu": 0.32,
        "dens": 2.7e3,
        "alpha": 2.2e-5,
        "kxx": 25.0,
        "cp": 210.0,
    },
    "TIM_pads": {
        "mat_id": 5,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSpreader": {
        "mat_id": 6,
        "E": 7.0e10,
        "nu": 0.29,
        "dens": 8.9e3,
        "alpha": 1.7e-5,
        "kxx": 400.0,
        "cp": 385.0,
    },
    "TIM2_pads": {
        "mat_id": 7,
        "E": 1.0e8,
        "nu": 0.30,
        "dens": 1.2e3,
        "alpha": 8.0e-5,
        "kxx": 2.0,
        "cp": 1500.0,
    },
    "HeatSink": {
        "mat_id": 8,
        "E": 6.9e10,
        "nu": 0.33,
        "dens": 2.7e3,
        "alpha": 2.4e-5,
        "kxx": 205.0,
        "cp": 900.0,
    },
    "Chiplet": {
        "mat_id": 9,
        "E": 1.0e11,
        "nu": 0.28,
        "dens": 2.33e3,
        "alpha": 2.6e-6,
        "kxx": 149.0,
        "cp": 700.0,
    },
}

# Loop over the dictionary and define materials in MAPDL
for mat_name, props in material_data.items():
    mid   = props["mat_id"]
    E     = props["E"]
    nu    = props["nu"]
    dens  = props["dens"]
    alpha = props["alpha"]
    k     = props["kxx"]
    cp    = props["cp"]

    # Reset temperature table and set a reference temperature (25°C)
    #mapdl.mptemp("", 0)
    #mapdl.mptemp(1, 25)

    # Define material properties
    mapdl.mp("EX",   mid, E)
    mapdl.mp("PRXY", mid, nu)
    mapdl.mp("DENS", mid, dens)
    mapdl.mp("ALPX", mid, alpha)
    mapdl.mp("KXX", mid, k)
    mapdl.mp("C", mid, cp)

print("Materials have been defined in MAPDL.")

##############################################################################
# 2) Assign Materials to Volumes via Their Components
##############################################################################
# Always start with selecting all volumes
mapdl.allsel()

# Substrate
mapdl.vsel("S", "Substrate")
mapdl.mat(material_data["Substrate"]["mat_id"])
mapdl.cmsel("ALL")

# C4_Bumps
mapdl.vsel("S", "C4_Bumps")
mapdl.mat(material_data["C4_Bumps"]["mat_id"])
mapdl.cmsel("ALL")

# Interposer
mapdl.vsel("S", "Interposer")
mapdl.mat(material_data["Interposer"]["mat_id"])
mapdl.cmsel("ALL")

# MicroBumps
mapdl.vsel("S", "MicroBumps")
mapdl.mat(material_data["MicroBumps"]["mat_id"])
mapdl.cmsel("ALL")

# TIM_pads
mapdl.vsel("S", "TIM_pads")
mapdl.mat(material_data["TIM_pads"]["mat_id"])
mapdl.cmsel("ALL")

# HeatSpreader
mapdl.vsel("S", "HeatSpreader")
mapdl.mat(material_data["HeatSpreader"]["mat_id"])
mapdl.cmsel("ALL")

# TIM2_pads
mapdl.vsel("S", "TIM2_pads")
mapdl.mat(material_data["TIM2_pads"]["mat_id"])
mapdl.cmsel("ALL")

# HeatSink (base and fins are in the same component)
mapdl.vsel("S", "HeatSink")
mapdl.mat(material_data["HeatSink"]["mat_id"])
mapdl.cmsel("ALL")

# Chiplets (each chiplet is a separate component)
for chip in ["Chiplet_1", "Chiplet_2", "Chiplet_3"]:
    mapdl.vsel("S", chip)
    mapdl.mat(material_data["Chiplet"]["mat_id"])
    mapdl.cmsel("ALL")

print("Material assignment via component selections is complete.")

##############################################################################
# 3) Visualize and Check
##############################################################################
# List all volumes with their details
print("Listing all volumes:")
mapdl.vlist()

# Display the component summary
#print("Component summary:")
#mapdl.cmstat()


In [ ]:
import ansys.aedt.core

In [ ]:
Imomin